# PVGIS ST-GNN - Enhanced MC Dropout + Deep Ensemble

Central notebook for this branch. All tunable values are in the config cell.

Workflow: single enhanced MC-Dropout run on W&B, Deep Ensemble W&B sweep over seeds, ensemble aggregation, post-hoc daytime/bin/anomaly analysis, interval-miss diagnostics, tables and figures.

## 1. Setup

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
from statistics import NormalDist

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.environ['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
os.chdir(REPO_ROOT)
print('repo root:', REPO_ROOT)


## 2. Central Config

In [ ]:
CONFIG = {
    'pvgis_dir': '/data/SentinelPV/pvgis_data/data/pvgis_summed_irradiance',
    'train_years': '2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018',
    'test_year': 2019,
    'anomaly_scores': 'outputs/pvgis_anomaly_2019_2005_2018_w15_q0975/pvgis_climatology_scores.csv',
    'target_variable': 'pv_power_output',

    'model_type': 'stgnn_enhanced_dropout',
    'feature_set': 'full',
    'seq_len': 24,
    'horizon': 1,
    'epochs': 60,
    'batch_size': 8,
    'lr': 1e-3,
    'dropout': 0.3,
    'device': 'cuda',
    'coverage_target': 0.95,
    'clc_eta': 9.0,

    'mc_seed': 1,
    'mc_samples': 30,
    'mc_run_name': 'enhanced_mc_dropout_seed1',
    'mc_out_dir': 'outputs/enhanced_mc_dropout/seed1',

    'ensemble_seeds': [1, 2, 3, 4, 5],
    'ensemble_out_root': 'outputs/enhanced_deep_ensemble/runs',
    'ensemble_pred_dir': 'outputs/enhanced_deep_ensemble/predictions',
    'ensemble_analysis_dir': 'outputs/enhanced_deep_ensemble/analysis',
    'ensemble_wandb_run_name': 'enhanced_deep_ensemble',

    'daytime_threshold': 10.0,
    'reference_peak_quantile': 0.99,
    'posthoc_chunksize': 1_000_000,

    'wandb_project': 'PhysiQ-PV',
    'wandb_entity': 'albertopedalino-politecnico-di-torino',
    'wandb_mode': 'online',
}

PIPELINE_TAG = 'mc_dropout'  # 'mc_dropout' or 'deep_ensemble'
RUN_PIPELINE = False          # True runs the selected pipeline end-to-end
RUN_WANDB_AGENT_HERE = True   # deep_ensemble: run the W&B agent in this notebook
LOG_POSTHOC_TO_WANDB = False

# Advanced/manual cells below stay disabled by default. The main entry point is
# PIPELINE_TAG + RUN_PIPELINE in the one-tag runner cell.
ADVANCED_RUN = {
    'mc_dropout': False,
    'mc_posthoc': False,
    'create_ensemble_sweep': False,
    'run_ensemble_agent_here': False,
    'deep_ensemble_analysis': False,
    'deep_ensemble_posthoc': False,
    'log_posthoc_to_wandb': LOG_POSTHOC_TO_WANDB,
}
PIPELINE_OUTPUTS = None

print(json.dumps(CONFIG, indent=2))
print('pipeline tag:', PIPELINE_TAG)
print('run pipeline:', RUN_PIPELINE)
print('advanced/manual run switches:', ADVANCED_RUN)


## 3. Commands

In [ ]:
def base_runner_args(cfg):
    return [
        '--pvgis-dir', cfg['pvgis_dir'],
        '--train-years', cfg['train_years'],
        '--test-year', str(cfg['test_year']),
        '--anomaly-scores', cfg['anomaly_scores'],
        '--target-variable', cfg['target_variable'],
        '--model-type', cfg['model_type'],
        '--feature-set', cfg['feature_set'],
        '--seq-len', str(cfg['seq_len']),
        '--horizon', str(cfg['horizon']),
        '--epochs', str(cfg['epochs']),
        '--batch-size', str(cfg['batch_size']),
        '--lr', str(cfg['lr']),
        '--dropout', str(cfg['dropout']),
        '--coverage-target', str(cfg['coverage_target']),
        '--device', cfg['device'],
        '--wandb-project', cfg['wandb_project'],
        '--wandb-entity', cfg['wandb_entity'],
    ]

def mc_dropout_cmd(cfg):
    return [
        sys.executable, 'scripts/run_pvgis_stgnn_forecasting.py',
        *base_runner_args(cfg),
        '--mc-dropout', '--mc-samples', str(cfg['mc_samples']),
        '--seed', str(cfg['mc_seed']),
        '--out-dir', cfg['mc_out_dir'],
        '--wandb', '--wandb-run-name', cfg['mc_run_name'],
    ]

def daytime_posthoc_cmd(predictions, out_dir, cfg, method_label=None, seed=None):
    return [
        sys.executable, 'scripts/analyze_pvgis_daytime_report.py',
        '--predictions', str(predictions), '--out-dir', str(out_dir),
        '--daytime-threshold', str(cfg['daytime_threshold']),
        '--coverage-target', str(cfg['coverage_target']),
        '--clc-eta', str(cfg['clc_eta']),
        '--reference-peak-quantile', str(cfg['reference_peak_quantile']),
        '--chunksize', str(cfg['posthoc_chunksize']),
        '--epochs', str(cfg['epochs']),
        '--dropout', str(cfg['dropout']),
        '--mc-samples', str(cfg['mc_samples']),
        '--method-label', method_label or PIPELINE_TAG,
        '--model-type', cfg['model_type'],
        '--feature-set', cfg['feature_set'],
        '--seed', str(seed if seed is not None else cfg['mc_seed']),
    ]

def interval_miss_cmd(predictions, out_dir, cfg):
    return [
        sys.executable, 'scripts/analyze_pvgis_interval_miss_distance.py',
        '--predictions', str(predictions), '--out-dir', str(out_dir),
        '--interval', 'pi', '--daytime-threshold', str(cfg['daytime_threshold']),
        '--anomaly-scores', cfg['anomaly_scores'],
    ]

def make_ensemble_sweep_config(cfg):
    return {
        'program': 'scripts/run_pvgis_stgnn_sweep_member.py',
        'method': 'grid',
        'metric': {'name': 'mae/global', 'goal': 'minimize'},
        'parameters': {'seed': {'values': cfg['ensemble_seeds']}},
        'command': [
            '${env}', '${interpreter}', '${program}',
            *base_runner_args(cfg),
            '--out-root', cfg['ensemble_out_root'],
            '--ensemble-dir', cfg['ensemble_pred_dir'],
            '${args}',
        ],
    }

def ensemble_analysis_cmd(cfg):
    analysis_dir = Path(cfg['ensemble_analysis_dir'])
    ref_seed = cfg['ensemble_seeds'][0]
    reference_predictions = Path(cfg['ensemble_out_root']) / f'seed{ref_seed}' / 'predictions.csv'
    return [
        sys.executable, 'scripts/analyze_pvgis_deep_ensemble.py',
        '--predictions-dir', cfg['ensemble_pred_dir'],
        '--out-dir', str(analysis_dir),
        '--coverage-target', str(cfg['coverage_target']),
        '--clc-eta', str(cfg['clc_eta']),
        '--reference-predictions', str(reference_predictions),
        '--wandb', '--wandb-project', cfg['wandb_project'],
        '--wandb-entity', cfg['wandb_entity'],
        '--wandb-run-name', cfg['ensemble_wandb_run_name'],
    ]

def run_if(enabled, cmd):
    print(' '.join(map(str, cmd)))
    if enabled:
        subprocess.run(cmd, check=True)
    else:
        print('not launching: switch is False')


## 4. Preflight

In [ ]:
checks = {
    'PVGIS dir': Path(CONFIG['pvgis_dir']).is_dir(),
    'anomaly scores': Path(CONFIG['anomaly_scores']).exists(),
    'runner wrapper': Path('scripts/run_pvgis_stgnn_forecasting.py').exists(),
    'sweep member': Path('scripts/run_pvgis_stgnn_sweep_member.py').exists(),
    'daytime posthoc': Path('scripts/analyze_pvgis_daytime_report.py').exists(),
    'interval posthoc': Path('scripts/analyze_pvgis_interval_miss_distance.py').exists(),
    'deep ensemble analyzer': Path('scripts/analyze_pvgis_deep_ensemble.py').exists(),
}
try:
    import physiq_pv.experiments.pvgis_stgnn_runner  # noqa: F401
    checks['runner importable'] = True
except Exception as exc:
    checks['runner importable'] = False
    print('runner import error:', exc)

for name, ok in checks.items():
    print(('OK     ' if ok else 'MISSING') + '  ' + name)


## 5. One-Tag Pipeline Runner

In [ ]:
def run_mc_dropout_pipeline(cfg):
    mc_out = Path(cfg['mc_out_dir'])
    predictions = mc_out / 'predictions.csv'
    daytime_dir = mc_out / 'posthoc_daytime'
    interval_dir = mc_out / 'posthoc_interval_miss'

    run_if(True, mc_dropout_cmd(cfg))
    run_if(True, daytime_posthoc_cmd(predictions, daytime_dir, cfg, method_label='mc_dropout', seed=cfg['mc_seed']))
    run_if(True, interval_miss_cmd(predictions, interval_dir, cfg))
    return {
        'scope': 'mc_dropout',
        'predictions': predictions,
        'daytime_dir': daytime_dir,
        'interval_dir': interval_dir,
    }

def run_deep_ensemble_pipeline(cfg, run_agent_here=True):
    import wandb
    os.environ['WANDB_MODE'] = cfg['wandb_mode']
    sweep_config = make_ensemble_sweep_config(cfg)
    sweep_id = wandb.sweep(
        sweep_config,
        project=cfg['wandb_project'],
        entity=cfg['wandb_entity'],
    )
    agent_ref = f"{cfg['wandb_entity']}/{cfg['wandb_project']}/{sweep_id}"
    print('sweep_id:', sweep_id)
    print('wandb agent ' + agent_ref)
    if run_agent_here:
        wandb.agent(
            sweep_id,
            project=cfg['wandb_project'],
            entity=cfg['wandb_entity'],
            count=len(cfg['ensemble_seeds']),
        )
    else:
        print('Run the agent command above, then rerun this cell with RUN_WANDB_AGENT_HERE=True or run the analysis cells manually.')
        return None

    analysis_dir = Path(cfg['ensemble_analysis_dir'])
    full_predictions = analysis_dir / 'deep_ensemble_predictions_full.csv'
    daytime_dir = analysis_dir / 'posthoc_daytime'
    interval_dir = analysis_dir / 'posthoc_interval_miss'

    run_if(True, ensemble_analysis_cmd(cfg))
    run_if(True, daytime_posthoc_cmd(full_predictions, daytime_dir, cfg, method_label='deep_ensemble', seed=0))
    run_if(True, interval_miss_cmd(full_predictions, interval_dir, cfg))
    return {
        'scope': 'deep_ensemble',
        'predictions': full_predictions,
        'daytime_dir': daytime_dir,
        'interval_dir': interval_dir,
    }

def run_selected_pipeline(tag, cfg):
    if tag == 'mc_dropout':
        return run_mc_dropout_pipeline(cfg)
    if tag == 'deep_ensemble':
        return run_deep_ensemble_pipeline(cfg, run_agent_here=RUN_WANDB_AGENT_HERE)
    raise ValueError("PIPELINE_TAG must be 'mc_dropout' or 'deep_ensemble'.")

PIPELINE_OUTPUTS = None
if RUN_PIPELINE:
    PIPELINE_OUTPUTS = run_selected_pipeline(PIPELINE_TAG, CONFIG)
else:
    print("Set RUN_PIPELINE=True and PIPELINE_TAG='mc_dropout' or 'deep_ensemble'.")


## 6. MC Dropout Run

In [ ]:
MC_OUT_DIR = Path(CONFIG['mc_out_dir'])
MC_PREDICTIONS = MC_OUT_DIR / 'predictions.csv'

run_if(ADVANCED_RUN['mc_dropout'], mc_dropout_cmd(CONFIG))


## 6. MC Dropout Post-Hoc

In [ ]:
MC_DAYTIME_POSTHOC_DIR = MC_OUT_DIR / 'posthoc_daytime'
MC_INTERVAL_POSTHOC_DIR = MC_OUT_DIR / 'posthoc_interval_miss'

run_if(ADVANCED_RUN['mc_posthoc'], daytime_posthoc_cmd(MC_PREDICTIONS, MC_DAYTIME_POSTHOC_DIR, CONFIG, method_label='mc_dropout', seed=CONFIG['mc_seed']))
run_if(ADVANCED_RUN['mc_posthoc'], interval_miss_cmd(MC_PREDICTIONS, MC_INTERVAL_POSTHOC_DIR, CONFIG))


## 7. Deep Ensemble W&B Sweep

In [ ]:
ensemble_sweep_config = make_ensemble_sweep_config(CONFIG)
print(json.dumps(ensemble_sweep_config, indent=2))

if ADVANCED_RUN['create_ensemble_sweep']:
    import wandb
    os.environ['WANDB_MODE'] = CONFIG['wandb_mode']
    sweep_id = wandb.sweep(
        ensemble_sweep_config,
        project=CONFIG['wandb_project'],
        entity=CONFIG['wandb_entity'],
    )
    agent_ref = f"{CONFIG['wandb_entity']}/{CONFIG['wandb_project']}/{sweep_id}"
    print('sweep_id:', sweep_id)
    print('run with : wandb agent ' + agent_ref)
    if ADVANCED_RUN['run_ensemble_agent_here']:
        wandb.agent(sweep_id, project=CONFIG['wandb_project'], entity=CONFIG['wandb_entity'], count=len(CONFIG['ensemble_seeds']))
else:
    print("Set ADVANCED_RUN['create_ensemble_sweep']=True to register the sweep.")
    print('Then run: wandb agent ' + CONFIG['wandb_entity'] + '/' + CONFIG['wandb_project'] + '/<sweep_id>')


## 8. Deep Ensemble Aggregation And Post-Hoc

In [ ]:
ENSEMBLE_ANALYSIS_DIR = Path(CONFIG['ensemble_analysis_dir'])
REFERENCE_SEED = CONFIG['ensemble_seeds'][0]
REFERENCE_PREDICTIONS = Path(CONFIG['ensemble_out_root']) / f'seed{REFERENCE_SEED}' / 'predictions.csv'
ENSEMBLE_FULL_PREDICTIONS = ENSEMBLE_ANALYSIS_DIR / 'deep_ensemble_predictions_full.csv'
ENSEMBLE_DAYTIME_POSTHOC_DIR = ENSEMBLE_ANALYSIS_DIR / 'posthoc_daytime'
ENSEMBLE_INTERVAL_POSTHOC_DIR = ENSEMBLE_ANALYSIS_DIR / 'posthoc_interval_miss'

run_if(ADVANCED_RUN['deep_ensemble_analysis'], ensemble_analysis_cmd(CONFIG))

run_if(ADVANCED_RUN['deep_ensemble_posthoc'], daytime_posthoc_cmd(ENSEMBLE_FULL_PREDICTIONS, ENSEMBLE_DAYTIME_POSTHOC_DIR, CONFIG, method_label='deep_ensemble', seed=0))
run_if(ADVANCED_RUN['deep_ensemble_posthoc'], interval_miss_cmd(ENSEMBLE_FULL_PREDICTIONS, ENSEMBLE_INTERVAL_POSTHOC_DIR, CONFIG))


## 9. Results Selector

In [ ]:
RESULT_SCOPE = (PIPELINE_OUTPUTS or {}).get('scope', PIPELINE_TAG)

if RESULT_SCOPE == 'mc_dropout':
    SELECTED_RUN_NAME = CONFIG['mc_run_name']
    SELECTED_PREDICTIONS = MC_PREDICTIONS
    SELECTED_DAYTIME_DIR = MC_DAYTIME_POSTHOC_DIR
    SELECTED_INTERVAL_DIR = MC_INTERVAL_POSTHOC_DIR
elif RESULT_SCOPE == 'deep_ensemble':
    SELECTED_RUN_NAME = CONFIG['ensemble_wandb_run_name']
    SELECTED_PREDICTIONS = ENSEMBLE_FULL_PREDICTIONS
    SELECTED_DAYTIME_DIR = ENSEMBLE_DAYTIME_POSTHOC_DIR
    SELECTED_INTERVAL_DIR = ENSEMBLE_INTERVAL_POSTHOC_DIR
else:
    raise ValueError(RESULT_SCOPE)

print('scope       :', RESULT_SCOPE)
print('predictions :', SELECTED_PREDICTIONS)
print('daytime dir :', SELECTED_DAYTIME_DIR)
print('interval dir:', SELECTED_INTERVAL_DIR)


## 10. Tables

In [ ]:
def read_csv_or_none(path):
    path = Path(path)
    return pd.read_csv(path) if path.exists() else None

RESULT_FILES = [
    'daytime_anomaly_overview.csv',
    'daytime_bin_summary.csv',
    'daytime_bin_anomaly_metrics.csv',
    'frequency_weighted_bin_summary.csv',
    'uncertainty_response.csv',
    'sharpness_overview.csv',
    'reference_production_peaks.csv',
]
results = {name: read_csv_or_none(SELECTED_DAYTIME_DIR / name) for name in RESULT_FILES}
interval_results = {
    'interval_miss_distance.csv': read_csv_or_none(SELECTED_INTERVAL_DIR / 'interval_miss_distance.csv'),
    'picp_curve.csv': read_csv_or_none(SELECTED_INTERVAL_DIR / 'picp_curve.csv'),
}

for name, df in {**results, **interval_results}.items():
    print((('OK  ' if df is not None else '--  ') + name) + (f'  {df.shape}' if df is not None else ''))

sharp = results['sharpness_overview.csv']
if sharp is not None:
    cols = [c for c in ['scope', 'count', 'picp', 'mae', 'rmse', 'mean_std', 'mpiw', 'nmpil', 'production_peak_nmpil', 'clc'] if c in sharp.columns]
    display(sharp[cols])
for name in ['daytime_bin_summary.csv', 'frequency_weighted_bin_summary.csv', 'uncertainty_response.csv', 'reference_production_peaks.csv']:
    df = results[name]
    if df is not None:
        display(df)


## 11. Reliability Diagnostic

In [ ]:
def reliability_table(predictions_csv, daytime_threshold=10.0, gamma=0.95, eta=9.0):
    path = Path(predictions_csv)
    if not path.exists():
        print('missing predictions:', path)
        return None, None
    want = {'y_true', 'y_pred_mean', 'y_pred', 'y_pred_std_raw', 'y_pred_std', 'lower_pi', 'upper_pi', 'solar_irradiance_poa_target'}
    df = pd.read_csv(path, usecols=lambda c: c in want)
    pred_col = 'y_pred_mean' if 'y_pred_mean' in df.columns else 'y_pred'
    std_col = 'y_pred_std_raw' if 'y_pred_std_raw' in df.columns else 'y_pred_std'
    day = df[df['solar_irradiance_poa_target'] > daytime_threshold].copy()
    need = ['y_true', pred_col, std_col, 'lower_pi', 'upper_pi']
    day = day[np.isfinite(day[need].to_numpy(float)).all(axis=1)]
    if day.empty:
        print('no valid daytime rows')
        return None, None
    y = day['y_true'].to_numpy(float)
    mu = day[pred_col].to_numpy(float)
    std = day[std_col].to_numpy(float)
    lower = day['lower_pi'].to_numpy(float)
    upper = day['upper_pi'].to_numpy(float)
    target_range = max(float(np.nanmax(y) - np.nanmin(y)), 1e-6)
    rmse = float(np.sqrt(np.mean((mu - y) ** 2)))
    normal = NormalDist()

    def metrics(lo, hi, name):
        inside = (y >= lo) & (y <= hi)
        picp = float(inside.mean())
        mpiw = float(np.mean(hi - lo))
        nmpil = mpiw / target_range
        clc = nmpil * (1.0 + np.exp(-eta * (picp - gamma)))
        return {'model': name, 'PICP': picp, 'MPIW': mpiw, 'NMPIL': nmpil, 'MPIW_RMSE': mpiw / rmse if rmse else np.nan, 'CLC': clc}

    z95 = normal.inv_cdf(0.5 + gamma / 2)
    sig0 = float(np.std(mu - y))
    table = pd.DataFrame([
        metrics(lower, upper, 'saved PI'),
        metrics(mu - z95 * std, mu + z95 * std, 'mean +/- z*std'),
        metrics(mu - z95 * sig0, mu + z95 * sig0, 'homoschedastic'),
    ])
    levels = np.array([0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99])
    empirical = []
    for level in levels:
        z = normal.inv_cdf(0.5 + level / 2)
        empirical.append(float(((y >= mu - z * std) & (y <= mu + z * std)).mean()))
    return table, pd.DataFrame({'nominal': levels, 'empirical': empirical})

rel_table, rel_curve = reliability_table(SELECTED_PREDICTIONS, CONFIG['daytime_threshold'], CONFIG['coverage_target'], CONFIG['clc_eta'])
if rel_table is not None:
    display(rel_table)
    display(rel_curve)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], 'k--', label='ideal')
    ax.plot(rel_curve['nominal'], rel_curve['empirical'], 'o-', label='mean +/- z*std')
    ax.set_xlabel('nominal coverage')
    ax.set_ylabel('empirical coverage')
    ax.set_title(f'Reliability diagram - {RESULT_SCOPE}')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()


## 12. Post-Hoc Figures

In [ ]:
def build_posthoc_figures(result_dir, title):
    result_dir = Path(result_dir)
    fig_dir = result_dir / 'figures'
    fig_dir.mkdir(parents=True, exist_ok=True)
    bin_cat = read_csv_or_none(result_dir / 'daytime_bin_anomaly_metrics.csv')
    freq = read_csv_or_none(result_dir / 'frequency_weighted_bin_summary.csv')
    unc = read_csv_or_none(result_dir / 'uncertainty_response.csv')
    if bin_cat is None:
        print('missing daytime_bin_anomaly_metrics.csv')
        return {}
    categories = [c for c in ['normal', 'rare_extreme', 'unusually_low_solar_potential', 'unusually_high_solar_potential', 'extreme_temperature_condition', 'extreme_wind_condition'] if c in set(bin_cat['category'])]
    plot_df = bin_cat[bin_cat['category'].isin(categories)].copy()
    x_labels = list(plot_df['bin'].drop_duplicates())
    x = np.arange(len(x_labels))
    width = 0.8 / max(len(categories), 1)
    paths = {}
    fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
    for idx, category in enumerate(categories):
        sub = plot_df[plot_df['category'] == category].set_index('bin').reindex(x_labels)
        offset = (idx - (len(categories) - 1) / 2) * width
        axes[0, 0].bar(x + offset, sub['mae'], width, label=category)
        axes[0, 1].bar(x + offset, sub['picp'], width, label=category)
        axes[1, 0].bar(x + offset, sub['nmpil'], width, label=category)
    axes[0, 1].axhline(CONFIG['coverage_target'], color='black', linestyle='--', linewidth=1)
    axes[0, 0].set_title('MAE by production bin')
    axes[0, 1].set_title('PICP by production bin')
    axes[1, 0].set_title('NMPIL by production bin')
    for ax in axes.flat[:3]:
        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=30, ha='right')
        ax.grid(axis='y', alpha=0.25)
    ax = axes[1, 1]
    if unc is not None and not unc.empty:
        u = unc.set_index('category').reindex([c for c in categories if c != 'normal']).dropna(how='all')
        ux = np.arange(len(u.index))
        ax.bar(ux - 0.18, u['mae_ratio_vs_normal'], 0.36, label='MAE ratio')
        ax.bar(ux + 0.18, u['std_ratio_vs_normal'], 0.36, label='std ratio')
        ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
        ax.set_xticks(ux)
        ax.set_xticklabels(u.index, rotation=30, ha='right')
        ax.set_title('Error vs uncertainty response')
        ax.grid(axis='y', alpha=0.25)
    else:
        ax.axis('off')
    axes[0, 1].legend(fontsize=8, ncol=2)
    axes[1, 1].legend(fontsize=8)
    fig.suptitle(title)
    path = fig_dir / 'posthoc_overview.png'
    fig.savefig(path, dpi=160)
    paths['posthoc_overview'] = path
    plt.show()
    if freq is not None and not freq.empty:
        fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
        f = freq.dropna(subset=['frequency_weighted_abs_picp_gap']).copy()
        ax.bar(f['category'], f['frequency_weighted_abs_picp_gap'])
        ax.set_title('Frequency-weighted absolute PICP gap')
        ax.set_ylabel('|PICP - target|')
        ax.tick_params(axis='x', rotation=30)
        ax.grid(axis='y', alpha=0.25)
        path = fig_dir / 'frequency_weighted_picp_gap.png'
        fig.savefig(path, dpi=160)
        paths['frequency_weighted_picp_gap'] = path
        plt.show()
    return paths

figure_paths = build_posthoc_figures(SELECTED_DAYTIME_DIR, f'{RESULT_SCOPE} post-hoc')
print({k: str(v) for k, v in figure_paths.items()})


## 13. Optional W&B Post-Hoc Logging

In [ ]:
if ADVANCED_RUN['log_posthoc_to_wandb']:
    import wandb
    run = wandb.init(project=CONFIG['wandb_project'], entity=CONFIG['wandb_entity'], name=f'posthoc_{SELECTED_RUN_NAME}', config={**CONFIG, 'result_scope': RESULT_SCOPE})
    try:
        for name, df in results.items():
            if df is not None:
                run.log({name.replace('.csv', ''): wandb.Table(dataframe=df)})
        for name, df in interval_results.items():
            if df is not None:
                run.log({name.replace('.csv', ''): wandb.Table(dataframe=df)})
        for name, path in figure_paths.items():
            run.log({f'figure/{name}': wandb.Image(str(path))})
        if sharp is not None and not sharp.empty:
            row = sharp[sharp['scope'].eq('overall_daytime')]
            if not row.empty:
                for col in ['picp', 'mae', 'rmse', 'mean_std', 'mpiw', 'nmpil', 'clc']:
                    if col in row.columns:
                        run.summary[f'posthoc/overall_daytime_{col}'] = float(row.iloc[0][col])
    finally:
        run.finish()
else:
    print("Set LOG_POSTHOC_TO_WANDB=True or ADVANCED_RUN['log_posthoc_to_wandb']=True to log tables and figures.")
